### Need for autograd

$$y = x^2$$

We have to write a program that find the derivative of this expression for a given x

$$\frac{dy}{dx} = 2x$$

Finding derivates for nested functions gets increasingly complex - chain rule

1. forward pass - compute the output of the network given an input
2. calculate loss
3. backward pass - compute gradients of the loss with respect to the parameters
4. update the parameters based on the gradients

Autograd- provides automatic differentiation for tensor operations. enables gradient computation which is essential for training ml models using optimization algorithms like gradient descent

In [75]:
import torch

#### Example

In [76]:
# `requires_grad=True` tells PyTorch that we want to find the derivative of that tensor
# PyTorch will track all the operations that we will perform on this tensor 

x = torch.tensor(3.0, requires_grad=True) # just one value, so it's a scalar

In [77]:
y = x**2

In [78]:
print(x)
print(y)

tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)


In [79]:
y.backward()

In [80]:
x.grad

tensor(6.)

#### Example

In [81]:
import math

In [82]:
# without PyTorch

def dz_dx(x):
    return 2 * x * math.cos(x**2)

dz_dx(3)

-5.466781571308061

In [83]:
# using autograd

x = torch.tensor(3.0, requires_grad=True)

y = x**2
z = torch.sin(y)

In [84]:
print(x)
print(y)
print(z)

tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)
tensor(0.4121, grad_fn=<SinBackward0>)


In [85]:
z.backward()
print(x.grad)

tensor(-5.4668)


#### Example

In [86]:
def bce(target, pred):
    epsilon = 1e-8
    pred = torch.clamp(pred, min=epsilon, max=(1-epsilon))
    return -(target*(torch.log(pred)) + (1-target)*(torch.log(1-pred)))

In [87]:
# without autograd

x = torch.tensor(6.7)
y = torch.tensor(0)

w = torch.tensor(1)
b = torch.tensor(0)

z = w*x + b
y_pred = torch.sigmoid(z)
loss = (y_pred, z)

# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


In [88]:
# with autograd

x = torch.tensor(6.7)
y = torch.tensor(0)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

z = w*x + b
y_pred = torch.sigmoid(z)

loss = bce(y, y_pred)

loss.backward()

print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


### Computation Graph

When we tell PyTorch to track gradients by setting `requires_grad=True` on a tensor, it silently builds a **Computation Graph** behind the scenes. A computation graph is a Directed Acyclic Graph (DAG) made up of interconnected nodes representing data and operations.

* **Leaf Nodes (Inputs):** starting points — raw input tensors we create ourselves, like data ($x$), weights ($w$), and biases ($b$). PyTorch calculates and stores gradients specifically for these leaf nodes.

* **Root Node (Output):** final result of the calculations — typically the final prediction or the calculated loss.

* **Intermediate Nodes:** by default, PyTorch does not save the gradients for intermediate nodes to conserve memory.

* **Edges (Operations):** the mathematical operations applied — PyTorch tracks these as a `grad_fn` (gradient function), which tells the engine exactly how to reverse the math later.

#### The Forward Pass (Calculating the Output)

When we run our mathematical equations normally, we are moving in the **forward direction** of the graph. We start at the leaf nodes, pass them through the operational edges, and calculate the values at each intermediate node until we reach the root node.

#### The Backward Pass (Calculating the Gradients)

When we call the `.backward()` function on our root node, PyTorch shifts into reverse, traversing the graph backward from the root down to the leaves.

* As it moves backward, it looks at the `grad_fn` of each operation and automatically applies the **chain rule**.

* It computes how much a tiny change in a leaf node would affect the final root node.

* Once it reaches the bottom, it populates the `.grad` attribute of the leaf tensors with the final derivative values.


### Vector as input

In [89]:
x = torch.tensor([3.4, 4.1, 5.5], requires_grad=True)
x

tensor([3.4000, 4.1000, 5.5000], requires_grad=True)

In [90]:
y = (x**2).mean()
y

tensor(19.5400, grad_fn=<MeanBackward0>)

In [91]:
y.backward()

x.grad

tensor([2.2667, 2.7333, 3.6667])

### Graph Control and Memory Management

* **Accumulation:** If we run the backward pass multiple times, PyTorch adds the new gradients to the existing ones. We have to manually clear them (using `.grad.zero_()`) before the next training step to avoid corrupting our calculations.

* **Disabling the Graph:** When we only need to make predictions and don't need to train the model, calculating gradients is a waste of memory. We can detach tensors from the computation graph entirely using `.detach()`, by setting `requires_grad_(False)`, or by wrapping our code in a `with torch.no_grad():` block.

#### Clearing gradients

In [92]:
x = torch.tensor(2.0, requires_grad=True)

y = x**2

y.backward()

print(x.grad)
x.grad.zero_()

tensor(4.)


tensor(0.)

#### Disabling gradient tracking

In [93]:
# requires_grad_(False)

x = torch.tensor(2.0, requires_grad=True)
print(x)

x.requires_grad_(False)
print(x)

y = x ** 2
print(y)
print(y.grad_fn)  # None
# y.backward()    # error

tensor(2., requires_grad=True)
tensor(2.)
tensor(4.)
None


In [94]:
# detach()
x = torch.tensor(2.0, requires_grad=True)

z = x.detach()
print(z)

y = x ** 2      # has grad_fn
y1 = z ** 2     # no grad_fn
print(y)
print(y1)

y.backward()    # works
print(x.grad)
# y1.backward() # error

tensor(2.)
tensor(4., grad_fn=<PowBackward0>)
tensor(4.)
tensor(4.)


In [95]:
# torch.no_grad()
x = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = x ** 2

print(y)
# y.backward()  # error

# without the with block, tracking is back
y = x ** 2
y.backward()
print(x.grad)

tensor(4.)
tensor(4.)
